# Auto AI Translator - Ultimate Edition (FastAPI + Streamlit)

Run this notebook to start the system.

**Architecture:**
*   **Backend**: FastAPI (`api.py`) runs on port 8000.
*   **Frontend**: Streamlit (`frontend.py`) runs on port 8501 and connects to Backend.
*   **Tunnel**: **Ngrok** exposes the Frontend to the internet.

In [1]:
!git clone --single-branch --branch monitor-duplicates-feature-12433170683288839432 https://github.com/Brian071/NewsScrapping.git temp_repo
!cp -r temp_repo/* .
!rm -rf temp_repo

Cloning into 'temp_repo'...
remote: Enumerating objects: 360, done.
remote: Counting objects: 100% (268/268), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 360 (delta 192), reused 169 (delta 108), pack-reused 92 (from 1)
Receiving objects: 100% (360/360), 533.00 KiB | 6.34 MiB/s, done.
Resolving deltas: 100% (227/227), done.


In [2]:
# 1. Install Dependencies
!pip install -r requirements.txt
!pip install streamlit --upgrade
!pip install pyngrok
!playwright install chromium
!playwright install-deps chromium

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 72.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.3/453.3 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

(node:731) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
164.7 MiB [] 0% 0.0s164.7 MiB [] 0% 38.6s164.7 MiB [] 0% 17.8s164.7 MiB [] 0% 14.4s164.7 MiB [] 0% 8.2s164.7 MiB [] 1% 4.5s164.7 MiB [] 2% 3.4s164.7 MiB [] 3% 2.9s164.7 MiB [] 4% 3.2s164.7 MiB [] 5% 2.8s164.7 MiB [] 6% 2.5s164.7 MiB [] 7% 2.2s164.7 MiB [] 8% 2.1s164.7 MiB [] 9% 2.0s164.7 MiB [] 10% 2.0s164.7 MiB [] 11% 1.9s164.7 MiB [] 12% 1.8s164.7 MiB [] 13% 1.8s164.7 MiB [] 14% 1.7s164.7 MiB [] 15% 1.7s164.7 MiB [] 16% 1.8s164.7 MiB [] 17% 1.7s164.7 MiB [] 18% 1.6s164.7 MiB [] 19% 1.6s164.7 MiB [] 20% 1.6s164.7 MiB [] 21% 1.5s164.7 MiB [] 22% 1.5s164.7 MiB [] 24% 1.4s164.7 MiB [] 25% 1.4s164.7 MiB [] 26% 1.3s164.7 MiB [] 28% 1.3s164.7 MiB [] 29% 1.3s164.7 MiB [] 30% 1.2s164.7 MiB [] 31% 1.

In [3]:
# 2. Google Authentication (REQUIRED for Google Sheets)
from google.colab import auth
auth.authenticate_user()
print("Authenticated with Google!")

Authenticated with Google!


In [ ]:
import os
import time
from google.colab import userdata
from pyngrok import ngrok

# 3. Run Backend (FastAPI) in background with logging
print("Starting Backend API...")
get_ipython().system_raw('uvicorn api:app --host 0.0.0.0 --port 8000 > api.log 2>&1 &')

# Wait for API to start
time.sleep(5)

# Check if API is running (Optional Debug)
!ps aux | grep uvicorn

# 4. Setup Ngrok
try:
    # Get Token from Colab Secrets (Name: NGROK)
    ngrok_token = userdata.get('NGROK')
    ngrok.set_auth_token(ngrok_token)

    # Start Tunnel for Streamlit (Port 8501)
    public_url = ngrok.connect(8501).public_url
    print(f"🚀 App is running! Click here: {public_url}")

except Exception as e:
    print(f"❌ Error connecting to Ngrok: {e}")
    print("Make sure you have added 'NGROK' to Colab Secrets.")
    print("Printing API Log for debugging:")
    !cat api.log

# 5. Run Frontend (Streamlit)
!python -m streamlit run frontend.py --server.headless true

Starting Backend API...
root        2759  0.0  0.0   7372  3392 ?        S    06:59   0:00 /bin/bash -c ps aux | grep uvicorn
root        2761  0.0  0.0   6480  2368 ?        S    06:59   0:00 grep uvicorn
🚀 App is running! Click here: https://trochal-postcephalic-manuela.ngrok-free.dev





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.170.248.119:8501

2026-01-30 07:01:20.025960: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-30 07:01:20.033129: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-30 07:01:20.054930: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769756480.096325    3238 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769756480.108930    3238 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS w